# Property Investment Advisor

A classification model to decide whether the property is worth buying or not

In [51]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_curve, roc_auc_score, auc
import dagshub
import os
from dotenv import load_dotenv
import mlflow
from mlflow.data import from_pandas

In [52]:
data = pd.read_csv('../Datasets/Property_Investment.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 30 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              250000 non-null  int64  
 1   State                           250000 non-null  object 
 2   City                            250000 non-null  object 
 3   Locality                        250000 non-null  object 
 4   Property_Type                   250000 non-null  object 
 5   BHK                             250000 non-null  int64  
 6   Size_in_SqFt                    250000 non-null  int64  
 7   Price_per_SqFt_in_Lakhs         250000 non-null  float64
 8   Price_in_Lakhs                  250000 non-null  float64
 9   Year_Built                      250000 non-null  int64  
 10  Furnished_Status                250000 non-null  object 
 11  Direction_Facing                250000 non-null  object 
 12  Floor_No        

In [53]:
data.head()

,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_per_SqFt_in_Lakhs,Price_in_Lakhs,Year_Built,Furnished_Status,Direction_Facing,Floor_No,Total_Floors,Age_of_Property,Owner_Type,Availability_Status,Total_Nearby_Schools,Total_Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Clubhouse,Garden,Gym,Playground,Pool,Amenity_Score,Future_Price_5Y,Invest
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,0.103,489.76,1990,Furnished,West,22,1,35,Owner,Yes,10,3,High,No,No,1,1,1,1,1,0.620,912.302247,No
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,0.083,195.52,2008,Unfurnished,North,21,20,17,Builder,No,8,1,Low,No,Yes,1,1,1,1,1,0.606,393.467805,Yes
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,0.050,183.79,1997,Semi-furnished,South,19,27,28,Broker,Yes,9,8,Low,Yes,No,1,0,1,1,1,0.526,314.467048,No
3,4,Rajasthan,Jodhpur,Locality_393,Independent House,2,2741,0.110,300.29,1991,Furnished,North,21,26,34,Builder,Yes,5,7,High,Yes,Yes,1,1,1,1,1,0.900,641.611647,No
4,5,Rajasthan,Jaipur,Locality_466,Villa,4,4823,0.038,182.90,2002,Semi-furnished,East,3,2,23,Builder,Yes,4,9,Low,No,Yes,1,1,1,1,1,0.766,313.898344,Yes


##  Feature Selection using Mutual Information

In [54]:
"""X_encoded = X.copy()

for col in X_encoded.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col])

mi_scores = mutual_info_classif(X_encoded, y)

mi_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'MI_Score': mi_scores
})

mi_df = mi_df.sort_values(by='MI_Score', ascending=False)

print(mi_df)"""

"X_encoded = X.copy()\n\nfor col in X_encoded.select_dtypes(include=['object']).columns:\n    le = LabelEncoder()\n    X_encoded[col] = le.fit_transform(X_encoded[col])\n\nmi_scores = mutual_info_classif(X_encoded, y)\n\nmi_df = pd.DataFrame({\n    'Feature': X_encoded.columns,\n    'MI_Score': mi_scores\n})\n\nmi_df = mi_df.sort_values(by='MI_Score', ascending=False)\n\nprint(mi_df)"

In [55]:
"""mi_df.sort_values(by='MI_Score').plot(
    x='Feature',
    y='MI_Score',
    kind='barh',
    figsize=(8,6)
)

plt.title("Feature Importance (Information Gain)")
plt.show()"""

'mi_df.sort_values(by=\'MI_Score\').plot(\n    x=\'Feature\',\n    y=\'MI_Score\',\n    kind=\'barh\',\n    figsize=(8,6)\n)\n\nplt.title("Feature Importance (Information Gain)")\nplt.show()'

In [56]:
# selected_features = selected_features = mi_df[mi_df['MI_Score'] > 0]['Feature'].tolist()

## Assigning Feature and Target variable

In [57]:
X = data.drop(columns=['Year_Built', 'Amenity_Score', 'State', 'Owner_Type', 'Future_Price_5Y', 'Price_per_SqFt_in_Lakhs', 'Total_Floors', 'Floor_No', 'ID', 'Availability_Status', 'Invest'])
y = data['Invest']

In [58]:
selected_features = X.columns

rf_tar_var = []
rf_num_var = []
rf_cat_var = []
rf_ord_var = []
rf_bin_var = []

for var in selected_features:

    # Drop NA for safety when checking unique values
    unique_vals = data[var].dropna().unique()

    # ✅ Step 1: Detect Binary (0/1)
    if set(unique_vals).issubset({0, 1}):
        rf_bin_var.append(var)

    # ✅ Step 2: Numerical / Ordinal
    elif (data[var].dtype == 'float') or (data[var].dtype == 'int'):
        rf_num_var.append(var)

    # ✅ Step 3: Categorical
    elif data[var].dtype == 'object':
        if var in ['City', 'Locality']:
            rf_tar_var.append(var)
        elif var == 'Public_Transport_Accessibility':
            rf_ord_var.append(var)
        else:
            rf_cat_var.append(var)

print(rf_tar_var)
print(rf_num_var)
print(rf_cat_var)
print(rf_ord_var)
print(rf_bin_var)

['City', 'Locality']
['BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Age_of_Property', 'Total_Nearby_Schools', 'Total_Nearby_Hospitals']
['Property_Type', 'Furnished_Status', 'Direction_Facing', 'Parking_Space', 'Security']
['Public_Transport_Accessibility']
['Clubhouse', 'Garden', 'Gym', 'Playground', 'Pool']


# Data Preprocessing and Model Pipeline

### Target Encoding: City and Locality together.

In [59]:
class CityLocalityTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, smoothing=10):
        self.smoothing = smoothing
        self.mapping_ = None
        self.global_mean_ = None

    def fit(self, X, y):
        X = X.copy()

        # ✅ Convert target to numeric
        y_numeric = pd.Series(y).map({'Yes': 1, 'No': 0})

        # Combine features
        X['City_Locality'] = X['City'].astype(str) + "_" + X['Locality'].astype(str)

        # Global mean (P(Yes))
        self.global_mean_ = y_numeric.mean()

        # Compute stats
        stats = (
            pd.DataFrame({'target': y_numeric, 'feature': X['City_Locality']})
            .groupby('feature')['target']
            .agg(['mean', 'count'])
        )

        # Apply smoothing
        smooth = (
            (stats['count'] * stats['mean'] + self.smoothing * self.global_mean_)
            / (stats['count'] + self.smoothing)
        )

        self.mapping_ = smooth.to_dict()

        return self

    def transform(self, X):
        X = X.copy()

        # Combine again
        X['City_Locality'] = X['City'].astype(str) + "_" + X['Locality'].astype(str)

        # Map encoding
        X['City_Locality_TE'] = X['City_Locality'].map(self.mapping_)

        # Handle unseen values
        X['City_Locality_TE'] = X['City_Locality_TE'].fillna(self.global_mean_)

        # Drop original columns
        X = X.drop(columns=['City', 'Locality', 'City_Locality'])

        return X

### Data Preprocessing Pipeline

In [60]:
city_locality_pipeline = Pipeline([
    ('target_encoder', CityLocalityTargetEncoder(smoothing=20))
])

binary_pipeline = 'passthrough'

ordinal_pipeline = Pipeline([
  ('ordinal', OrdinalEncoder())
])

num_pipeline = Pipeline([
    ('num', StandardScaler())
])

cat_pipeline = Pipeline([
    ('cat', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('city_locality_te', city_locality_pipeline, rf_tar_var),
        ('num', num_pipeline, rf_num_var),
        ('cat', cat_pipeline, rf_cat_var),
        ('ordinal', ordinal_pipeline, rf_ord_var),
        ('bin', binary_pipeline, rf_bin_var)
    ],
    remainder='drop'
)

### Decision Tree Classifier

In [61]:
dt_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', DecisionTreeClassifier(
        random_state=42
    ))
])

dt_tags = {
    "model": "Decision Tree Classifier"
}

### Random Forest Classifier Pipeline

In [62]:
rf_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100, 
        random_state=42
    ))
])

rf_tags = {
    "model": "Random Forest Classifier"
}

# Model Performance Visualization 

In [63]:
def create_roc_curve(model, model_name, X_test, y_test):
    
    y_test_num = pd.Series(y_test).map({'Yes': 1, 'No': 0})
    
    # Get predicted probabilities for the positive class
    y_prob = model.predict_proba(X_test)[:, 1]

    # Compute ROC curve points and AUC score
    fpr, tpr, _ = roc_curve(y_test_num, y_prob)
    roc_auc = auc(fpr, tpr)

    # ── Plot ──────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(fpr, tpr, color="steelblue", lw=2,
            label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], color="gray", lw=1,
            linestyle="--", label="Random Classifier")

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve — {model_name}")
    ax.legend(loc="lower right")

    plt.tight_layout()

    # ── Save locally ─────────────────────────
    safe_name = model_name.replace(" ", "_")
    save_dir  = os.path.join("Images", "Classification", safe_name)
    os.makedirs(save_dir, exist_ok=True)

    save_path = os.path.join(save_dir, "ROC_Curve.png")
    fig.savefig(save_path, dpi=150)
    plt.close(fig)

    print(f"ROC Curve saved → {save_path}  (AUC: {roc_auc:.3f})")
    return save_path, roc_auc

# Splitting into Training and Testing Data

In [64]:
dataset_params = {
  'random_state':42,
  'test_size':0.2
}

X_train, X_test, y_train, y_test = train_test_split(
  X, y, random_state = dataset_params['random_state'], test_size=dataset_params['test_size']
)

# Model Tracking using MLFlow and DagsHub

In [65]:
models = [
    (
        "Decision Tree Classifier",
        dt_pipeline,
        (X_train, y_train),
        (X_test, y_test),
        dt_tags
    ),
    (
        "Random Forest Classifier",
        rf_pipeline,
        (X_train, y_train),
        (X_test, y_test),
        rf_tags
    )
    
]

In [66]:
model_tags = []
model_reports = []
model_roc_paths = []
model_roc_auc_scores = []

#-----------------------------
# Training Loop
#-----------------------------

for model_name, model, train_set, test_set, tags in models:
    model_tags.append(tags)

    X_train, y_train = train_set
    X_test, y_test = test_set

    model.fit(X_train, y_train)
    print(f"{model_name} Model Trained")

    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    model_reports.append(report)
    
    roc_path, roc_auc = create_roc_curve(model, model_name, X_test, y_test)
    model_roc_auc_scores.append(roc_auc)
    model_roc_paths.append(roc_path)


Decision Tree Classifier Model Trained
ROC Curve saved → Images\Classification\Decision_Tree_Classifier\ROC_Curve.png  (AUC: 0.900)
Random Forest Classifier Model Trained
ROC Curve saved → Images\Classification\Random_Forest_Classifier\ROC_Curve.png  (AUC: 0.984)


# Dagshub Connection

In [67]:
dagshub.init(
    repo_owner='JS-Tharun', 
    repo_name='Real-Estate-Investment-Advisor', 
    mlflow=True
)

Initialized MLflow to track repo "JS-Tharun/Real-Estate-Investment-Advisor"

Repository JS-Tharun/Real-Estate-Investment-Advisor initialized!

In [68]:
# -------------------------
# MLFLOW LOGGING LOOP
# -------------------------
load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('DAGSHUB_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('DAGSHUB_PASSWORD')}"

mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME_ADV"])
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = model_reports[i]
    model_roc_auc = model_roc_auc_scores[i]
    roc_path = model_roc_paths[i]

    X_train, y_train = element[2]
    X_test, y_test = element[3]

    dataset = from_pandas(X_train, name="X_train_dataset")
    model_tag = model_tags[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_input(dataset, context='training_data')
        print("Input Data Logged")

        mlflow.set_tags(model_tag)
        print("Model Tags Logged")

        mlflow.log_param('model_name', model_name)
        mlflow.log_params(model.named_steps['model'].get_params())
        print("Model Paramenters Logged")

        mlflow_metrics = {}
        for label, metrics in report.items():
            if isinstance(metrics, dict):  # class-wise or avg metrics
                for metric_name, value in metrics.items():
                    if metric_name != "support":  # optional: skip support (not a metric)
                        key = f"{label}_{metric_name}"
                        mlflow_metrics[key] = float(value)
            else:
                # accuracy case (single value)
                mlflow_metrics[label] = float(metrics)
        mlflow.log_metrics(mlflow_metrics)
    
        mlflow_metrics["roc_auc"] = float(model_roc_auc)   # ← AUC as a metric
        mlflow.log_metrics(mlflow_metrics)
        print("Metrics Logged")

        mlflow.sklearn.log_model(
            model,
            model_name
        )
        print('Model Logged')

        mlflow.log_artifact(roc_path, artifact_path="plots")
        print("ROC Curve Artifact Logged")
        print("ROC Curve Logged")

Input Data Logged
Model Tags Logged
Model Paramenters Logged


2026/04/15 12:13:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Metrics Logged


2026/04/15 12:13:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model Logged
ROC Curve Artifact Logged
ROC Curve Logged
🏃 View run Decision Tree Classifier at: https://dagshub.com/JS-Tharun/Real-Estate-Investment-Advisor.mlflow/#/experiments/2/runs/0cfdaa53925c4c7584dac754dff721e3
🧪 View experiment at: https://dagshub.com/JS-Tharun/Real-Estate-Investment-Advisor.mlflow/#/experiments/2
Input Data Logged
Model Tags Logged
Model Paramenters Logged


2026/04/15 12:13:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Metrics Logged


2026/04/15 12:14:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model Logged
ROC Curve Artifact Logged
ROC Curve Logged
🏃 View run Random Forest Classifier at: https://dagshub.com/JS-Tharun/Real-Estate-Investment-Advisor.mlflow/#/experiments/2/runs/7ea9a1831ad7471f9660d745322844c5
🧪 View experiment at: https://dagshub.com/JS-Tharun/Real-Estate-Investment-Advisor.mlflow/#/experiments/2


ROC Curve

In [69]:
"""fpr, tpr, _ = roc_curve(y_test_num, probs)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()"""

'fpr, tpr, _ = roc_curve(y_test_num, probs)\n\nplt.plot(fpr, tpr)\nplt.xlabel("False Positive Rate")\nplt.ylabel("True Positive Rate")\nplt.title("ROC Curve")\nplt.show()'

In [70]:
"""preds = model.predict(X_test)

confidence = model.predict_proba(X_test).max(axis=1)

results = pd.DataFrame({
    "Prediction": preds,
    "Confidence": confidence
})"""

'preds = model.predict(X_test)\n\nconfidence = model.predict_proba(X_test).max(axis=1)\n\nresults = pd.DataFrame({\n    "Prediction": preds,\n    "Confidence": confidence\n})'